In [24]:
# ============================================
# TEMPLATE PREPROCESSING - REUSABLE
# ============================================

# Data manipulation and numerical computing
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Train/test split and scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Evaluation metrics
from sklearn.metrics import (classification_report, 
                             roc_auc_score, 
                             ConfusionMatrixDisplay)

# Decision Tree and Random Forest models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import AdaBoostClassifier

# --------------------------------------------
# 1. LOAD DATA
# --------------------------------------------
df = pd.read_csv(r"C:\Users\Giada Jenny Qafalia\Desktop\develhope\TEORIA\data-science-ml-portfolio\data\WA_Fn-UseC_-Telco-Customer-Churn.xls")

# --------------------------------------------
# 2. CLEANING
# --------------------------------------------
# TotalCharges has hidden spaces ' ' → convert to numeric
# errors='coerce' turns invalid values into NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop rows with NaN (only 11 rows affected)
df.dropna(inplace=True)

# Drop customer ID → not a predictive feature
df.drop(columns=['customerID'], inplace=True)

# Encode target variable as binary
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# --------------------------------------------
# 3. DEFINE FEATURES
# --------------------------------------------
# Continuous columns → will be scaled
continuous_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Categorical columns → will be one-hot encoded
# Automatically selects all columns except continuous and target
categorical_cols = [col for col in df.columns 
                    if col not in continuous_cols + ['Churn']]

# --------------------------------------------
# 4. ONE-HOT ENCODING
# --------------------------------------------
# drop_first=True avoids the dummy variable trap
# e.g. Gender: Male/Female → only Female column kept
df_encoded = pd.get_dummies(df, 
                             columns=categorical_cols, 
                             drop_first=True)

# --------------------------------------------
# 5. SPLIT X AND y
# --------------------------------------------
# X → all features (everything except target)
# y → target variable (Churn)
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

# --------------------------------------------
# 6. TRAIN/TEST SPLIT
# --------------------------------------------
# test_size=0.2 → 80% train, 20% test
# stratify=y → preserves class balance in both sets
# random_state=42 → reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 7. ── SCALING — obbligatorio per K-Means ───────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Preprocessing complete")
print(f"   Dataset: {X.shape}")
print(f"   Churn rate: {df['Churn'].mean():.2%}")

✅ Preprocessing complete
   Dataset: (7032, 30)
   Churn rate: 26.58%


In [25]:
# base estimator → stump (max_depth=1)
base = DecisionTreeClassifier(max_depth=1)

# model
ada = AdaBoostClassifier(estimator=base, 
                         n_estimators=75, 
                         learning_rate=1.0,
                         random_state=42)

In [26]:
#FIT
ada.fit(X_train, y_train)

#PREDICT
y_pred = ada.predict(X_test)
y_proba = ada.predict_proba(X_test)[:,1]

print("=== AdaBoost ===")
print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.3f}")

=== AdaBoost ===
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.64      0.50      0.56       374

    accuracy                           0.79      1407
   macro avg       0.74      0.70      0.71      1407
weighted avg       0.78      0.79      0.79      1407

AUC-ROC: 0.841


## Confronto Modelli — Telco Churn

| Modello      | AUC-ROC | Recall | F1   |
|--------------|---------|--------|------|
| LogReg       | 0.842   | 0.56   | 0.59 |
| KNN (K=27)   | 0.828   | 0.59   | 0.60 |
| LDA          | 0.828   | 0.56   | 0.59 |
| RF           | 0.832   | 0.40   | 0.50 |
| AdaBoost     | 0.841   | 0.50   | 0.56 |

## Perché il Recall è la metrica prioritaria

In un contesto di churn detection, il costo di un **False Negative**
(mancata identificazione di un cliente a rischio) supera 
significativamente il costo di un **False Positive** 
(cliente fedele erroneamente segnalato).

- **False Negative** → nessun intervento → cliente perso
  → costi stimati: recruiting, onboarding, perdita di know-how
  → fino al 200% dello stipendio annuo (contesto HR)
- **False Positive** → azione di retention non necessaria
  → costi: retention package, tempo HR → comparativamente bassi

Quando il costo FN >> costo FP, il **Recall è la metrica da massimizzare**.

## Quando usare AdaBoost

AdaBoost ottiene un AUC-ROC di 0.841 — competitivo con la 
Regressione Logistica e tra i più alti del confronto. Tuttavia, 
il Recall (0.50) è inferiore a KNN (0.59), rendendolo meno 
adatto quando minimizzare i clienti a rischio non identificati 
è l'obiettivo principale.

AdaBoost è una scelta solida quando:
- È richiesto un AUC-ROC elevato con tuning minimo
- Il dataset è pulito e privo di outlier (AdaBoost è sensibile al rumore)
- Non è necessaria l'interpretabilità dei singoli stump

In [27]:
#XGBoost
from xgboost import XGBClassifier

#define model
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42
)

#3 fit
xgb.fit(X_train, y_train)

#4 Predict
y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:,1]

#5 Evaluate
print("=== XGBoost ===")
print(classification_report(y_test, y_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.3f}")

=== XGBoost ===
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1033
           1       0.63      0.54      0.58       374

    accuracy                           0.79      1407
   macro avg       0.73      0.71      0.72      1407
weighted avg       0.78      0.79      0.79      1407

AUC-ROC: 0.837


## Confronto Modelli — Telco Churn (Aggiornato)

| Modello      | AUC-ROC | Recall | F1   |
|--------------|---------|--------|------|
| LogReg       | 0.842   | 0.56   | 0.59 |
| AdaBoost     | 0.841   | 0.50   | 0.56 |
| XGBoost      | 0.837   | 0.54   | 0.58 |
| RF           | 0.832   | 0.40   | 0.50 |
| KNN (K=27)   | 0.828   | 0.59   | 0.60 | ← miglior Recall
| LDA          | 0.828   | 0.56   | 0.59 |
| SVM          | 0.783   | 0.48   | 0.55 |

## Osservazioni — Modelli Semplici vs Complessi

Su questo dataset i modelli semplici (LogReg, KNN) ottengono 
Recall superiore rispetto ai modelli ensemble complessi (XGBoost, RF).

Questo comportamento è spiegabile con due caratteristiche del dataset:
- **Dimensione limitata** (7.032 righe) → i modelli complessi hanno 
  meno dati per esprimere il loro potenziale e rischiano overfitting
- **Classi sbilanciate** (73% No churn, 27% Churn) → i modelli 
  complessi tendono a specializzarsi sulla classe maggioritaria, 
  penalizzando il Recall sulla classe minoritaria

## XGBoost vs AdaBoost

| Caratteristica     | AdaBoost         | XGBoost                  |
|--------------------|------------------|--------------------------|
| Base learner       | Stump (depth=1)  | Albero profondo (depth 3-6) |
| Correzione errori  | Pesi campioni    | Residui                  |
| Ordine             | Sequenziale      | Sequenziale              |
| Regularizzazione   | No               | Sì (L1, L2)              |
| Robustezza outlier | Bassa            | Alta                     |
| Anno               | 1995             | 2016                     |
| AUC-ROC (Telco)    | 0.841            | 0.837                    |

## Quando usare XGBoost

XGBoost è la scelta preferibile quando:
- Il dataset è **grande** (100k+ righe) e le feature sono numerose
- È richiesta **alta performance** con tuning sistematico 
  degli iperparametri
- Il dataset è **bilanciato** o viene applicato class weighting


Su dataset piccoli e sbilanciati come Telco Churn, modelli più 
semplici come LogReg o KNN rimangono competitive baseline 
difficili da battere senza tuning approfondito.